# 09 — TMDB Keyword × SCL Enrichment

Joins the keyword enrichment table (`04`) with the SCL joined lexicon (`08`) to add
sentiment/valence data to every keyword that can be matched.

**Inputs**
- `output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv` — 84,842 keywords with taxonomy + movie_count
- `data/lexicons/scl_joined.csv` — 57,005 SCL terms (VAD + OPP + NMA)

**New columns added**

| Column | Description |
|---|---|
| `scl_valence` | Hierarchy-selected valence (OPP → NMA → VAD) |
| `scl_valence_source` | Which lexicon supplied the score (`vad` / `opp` / `nma`) |
| `scl_in_vad` | Present in NRC VAD |
| `scl_in_opp` | Present in SCL-OPP (opposing-polarity phrases) |
| `scl_in_nma` | Present in SCL-NMA (negator/modal/adverb phrases) |
| `scl_is_scored` | Has any valence score |
| `scl_is_phrase_level` | Score came from OPP or NMA (phrase-level empirical) vs direct VAD |
| `scl_polarity` | `positive` / `negative` / `neutral` / `unscored` |
| `scl_composition_type` | Linguistic mechanism (direct / negation / degree_modifier / …) |
| `scl_vad_arousal` | VAD arousal (kept for reference) |
| `scl_vad_dominance` | VAD dominance (kept for reference) |
| `scl_composition_shift` | Phrase shift from constituent unigrams |
| `scl_shift_zscore` | Z-score of composition shift |

Analytics use `is_narrative` to focus on keywords with story-signal value.


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

KEYWORDS_FILE = Path("output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv")
SCL_FILE      = Path("data/lexicons/scl_joined.csv")
OUTPUT_FILE   = Path("output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv")


## Load

In [2]:
keywords = pd.read_csv(KEYWORDS_FILE)
scl      = pd.read_csv(SCL_FILE)

# Drop stale SCL columns if re-running
scl_cols = [c for c in keywords.columns if c.startswith("scl_")]
if scl_cols:
    keywords = keywords.drop(columns=scl_cols)

print(f"Keywords : {len(keywords):,}  columns: {list(keywords.columns)}")
print(f"SCL      : {len(scl):,}  columns: {list(scl.columns)}")


Keywords : 84,842  columns: ['tmdb_keyword_id', 'name', 'keyword_type', 'lexical_class', 'min_zipf_freq', 'is_narrative', 'movie_count', 'manually_reviewed']
SCL      : 57,005  columns: ['term', 'ngrams', 'in_vad', 'in_opp', 'in_nma', 'vad_valence', 'vad_arousal', 'vad_dominance', 'opp_valence', 'opp_pos', 'opp_freq', 'opp_is_hashtag', 'nma_valence', 'valence_count', 'valence_variance', 'valence_source', 'valence', 'negators_contained', 'modals_contained', 'adverbs_contained', 'computed_valence', 'composition_shift', 'abs_composition_shift', 'shift_zscore', 'shift_percentile', 'shift_strong', 'shift_very_strong', 'composition_type', 'generated_valence', 'generated_n_tokens']


## Join

In [3]:
# Normalise join key
keywords["_name_norm"] = keywords["name"].str.lower().str.strip()
scl["_term_norm"]      = scl["term"].str.lower().str.strip()

# Select SCL columns to bring in (rename with scl_ prefix)
SCL_KEEP = {
    "valence":           "scl_valence",
    "valence_source":    "scl_valence_source",
    "in_vad":            "scl_in_vad",
    "in_opp":            "scl_in_opp",
    "in_nma":            "scl_in_nma",
    "composition_type":  "scl_composition_type",
    "composition_shift": "scl_composition_shift",
    "shift_zscore":      "scl_shift_zscore",
    "shift_percentile":  "scl_shift_percentile",
}

scl_slim = (
    scl[["_term_norm"] + list(SCL_KEEP.keys())]
    .drop_duplicates("_term_norm")
    .rename(columns=SCL_KEEP)
)

keywords = keywords.merge(scl_slim, left_on="_name_norm", right_on="_term_norm", how="left")
keywords = keywords.drop(columns=["_name_norm", "_term_norm"])

print(f"After join: {keywords.shape}")


After join: (84842, 17)


## Derived flags

In [4]:
# Has any valence score
keywords["scl_is_scored"] = keywords["scl_valence"].notna()

# Phrase-level: OPP or NMA (empirically scored constructions vs direct VAD lookup)
keywords["scl_is_phrase_level"] = keywords["scl_valence_source"].isin(["opp", "nma"])

# Polarity bucket
def polarity(v):
    if pd.isna(v):  return "unscored"
    if v > 0:       return "positive"
    if v < 0:       return "negative"
    return "neutral"

keywords["scl_polarity"] = keywords["scl_valence"].map(polarity)

scored = keywords["scl_is_scored"].sum()
print(f"Scored keywords    : {scored:,} / {len(keywords):,} ({scored/len(keywords)*100:.1f}%)")
print()
print("Polarity breakdown:")
print(keywords["scl_polarity"].value_counts().to_string())
print()
print("Valence source:")
print(keywords["scl_valence_source"].value_counts().to_string())
print()
print("Phrase-level (OPP/NMA):", keywords["scl_is_phrase_level"].sum())


Scored keywords    : 12,664 / 84,842 (14.9%)

Polarity breakdown:
scl_polarity
unscored    72178
positive     6756
negative     5281
neutral       627

Valence source:
scl_valence_source
vad    11527
nma      712
opp      425

Phrase-level (OPP/NMA): 1137


## Tokenizer fallback — fill unscored keywords via unigram average

For keywords with no direct SCL match, split the keyword name into tokens and average
the hierarchy-selected valence of each token that exists as a unigram in the SCL lexicon.

- `scl_valence_source = "unigram_avg"` distinguishes filled scores from direct matches.
- `scl_generated_coverage` = fraction of tokens with a unigram valence (0–1). Higher is more reliable.
- Single-token keywords with no SCL match remain unscored (can't decompose further).


In [5]:
# Build unigram valence lookup from the full SCL joined lexicon
# (scl df is still in scope; 'valence' is the hierarchy-selected column; '_term_norm' added in join cell)
unigram_val = (
    scl[(scl["ngrams"] == 1) & scl["valence"].notna()]
    .drop_duplicates("_term_norm")
    .set_index("_term_norm")["valence"]
    .to_dict()
)
print(f"Unigrams available for fallback : {len(unigram_val):,}")

unscored_mask = ~keywords["scl_is_scored"]
print(f"Unscored keywords               : {unscored_mask.sum():,}")

# Vectorised: split name into tokens, look up each, average
def _tok_stats(name):
    tokens = str(name).lower().strip().split()
    vals = [unigram_val[t] for t in tokens if t in unigram_val]
    if not vals:
        return None, 0.0
    return float(sum(vals) / len(vals)), len(vals) / len(tokens)

tok = keywords.loc[unscored_mask, "name"].apply(
    lambda n: pd.Series(_tok_stats(n), index=["_tv", "_cov"])
)
filled = unscored_mask & tok["_tv"].notna()

keywords.loc[filled, "scl_valence"]           = tok.loc[filled[unscored_mask], "_tv"].values
keywords.loc[filled, "scl_valence_source"]    = "unigram_avg"
keywords.loc[filled, "scl_is_scored"]         = True
keywords.loc[filled, "scl_is_phrase_level"]   = False

# Polarity for newly filled rows
v = keywords.loc[filled, "scl_valence"]
keywords.loc[filled, "scl_polarity"] = (
    pd.cut(v, bins=[-float("inf"), -1e-9, 1e-9, float("inf")],
           labels=["negative", "neutral", "positive"])
    .astype(str)
)

# Generated coverage ratio (0–1) — NaN for directly scored rows
keywords["scl_generated_coverage"] = None
keywords.loc[filled, "scl_generated_coverage"] = tok.loc[filled[unscored_mask], "_cov"].values

print(f"Filled via tokenizer            : {filled.sum():,}")
print(f"Total scored after fallback     : {keywords['scl_is_scored'].sum():,} / {len(keywords):,} "
      f"({keywords['scl_is_scored'].mean()*100:.1f}%)")
print()
print("Valence source breakdown:")
print(keywords["scl_valence_source"].value_counts())


Unigrams available for fallback : 44,809
Unscored keywords               : 72,178


Filled via tokenizer            : 36,474
Total scored after fallback     : 49,138 / 84,842 (57.9%)

Valence source breakdown:
scl_valence_source
unigram_avg    36474
vad            11527
nma              712
opp              425
Name: count, dtype: int64


## Coverage analytics

In [6]:
# Coverage split: narrative vs non-narrative
cov = (
    keywords.groupby("is_narrative")
    .apply(lambda g: pd.Series({
        "total":        len(g),
        "scored":       g["scl_is_scored"].sum(),
        "pct_scored":   f"{g['scl_is_scored'].mean()*100:.1f}%",
        "positive":     (g["scl_polarity"] == "positive").sum(),
        "negative":     (g["scl_polarity"] == "negative").sum(),
        "phrase_level": g["scl_is_phrase_level"].sum(),
    }))
    .rename(index={True: "narrative", False: "non-narrative"})
)
print("Coverage by is_narrative:")
print(cov.to_string())


Coverage by is_narrative:
               total  scored pct_scored  positive  negative  phrase_level
is_narrative                                                             
non-narrative    177     162      91.5%       133        28             8
narrative      84665   48976      57.8%     30801     16897          1129


In [7]:
cov_type = (
    keywords.groupby("keyword_type")
    .apply(lambda g: pd.Series({
        "total":      len(g),
        "scored":     g["scl_is_scored"].sum(),
        "pct_scored": g["scl_is_scored"].mean() * 100,
        "avg_valence": g["scl_valence"].mean(),
    }))
    .sort_values("pct_scored", ascending=False)
)

fig = px.bar(
    cov_type.reset_index(),
    x="keyword_type", y="pct_scored", text_auto=".1f",
    title="SCL coverage % by keyword_type",
    labels={"keyword_type": "", "pct_scored": "% scored"},
    color="avg_valence",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
)
fig.show()
print(cov_type[["total","scored","pct_scored","avg_valence"]].round(3).to_string())


                    total   scored  pct_scored  avg_valence
keyword_type                                               
trope                18.0     18.0     100.000        0.132
object_motif        378.0    376.0      99.471        0.068
character_role     1028.0   1014.0      98.638        0.140
production_meta     110.0    101.0      91.818        0.205
genre_as_keyword     67.0     61.0      91.045        0.185
time_period         262.0    213.0      81.298        0.047
plot_event        11068.0   8835.0      79.825        0.118
location           1049.0    775.0      73.880        0.184
mythic_figure        12.0      8.0      66.667        0.027
theme_or_plot     70850.0  37737.0      53.263        0.087


In [8]:
# Coverage vs movie frequency — do popular keywords have better SCL coverage?
buckets = [0,1,2,5,10,25,50,100,500,float("inf")]
labels  = ["1","2","3-5","6-10","11-25","26-50","51-100","101-500","500+"]
kw_used = keywords[keywords["movie_count"] > 0].copy()
kw_used["bucket"] = pd.cut(kw_used["movie_count"], bins=buckets, labels=labels)

cov_freq = (
    kw_used.groupby("bucket", observed=True)
    .apply(lambda g: pd.Series({
        "keywords":   len(g),
        "scored":     g["scl_is_scored"].sum(),
        "pct_scored": g["scl_is_scored"].mean() * 100,
    }))
    .reset_index()
)

fig2 = px.bar(cov_freq, x="bucket", y="pct_scored", text_auto=".1f",
    title="SCL coverage % by movie frequency bucket",
    labels={"bucket": "Movies using keyword", "pct_scored": "% with SCL score"})
fig2.show()


## Valence distribution — narrative keywords only

In [9]:
narrative_scored = keywords[keywords["is_narrative"] & keywords["scl_is_scored"]]

fig3 = px.histogram(
    narrative_scored, x="scl_valence", color="scl_valence_source",
    nbins=50, barmode="overlay", opacity=0.75,
    title=f"Valence distribution — narrative keywords with SCL score (n={len(narrative_scored):,})",
    labels={"scl_valence": "Valence [-1, +1]", "scl_valence_source": "Lexicon"},
    color_discrete_map={"vad": "#4C78A8", "opp": "#F58518", "nma": "#54A24B"},
)
fig3.add_vline(x=0, line_dash="dash", line_color="grey")
fig3.show()

print(narrative_scored["scl_valence"].describe().round(4).to_string())


count    48976.0000
mean         0.0952
std          0.3828
min         -1.0000
25%         -0.1255
50%          0.1295
75%          0.3600
max          1.0000


In [10]:
# Valence distribution by keyword_type — narrative only
fig4 = px.box(
    narrative_scored,
    x="keyword_type", y="scl_valence",
    points="outliers",
    title="Valence by keyword_type — narrative keywords",
    labels={"keyword_type": "", "scl_valence": "Valence [-1, +1]"},
)
fig4.add_hline(y=0, line_dash="dash", line_color="grey")
fig4.update_layout(xaxis_tickangle=-30)
fig4.show()


In [11]:
# Most positive and most negative narrative keywords (min 10 movies)
active_narrative = narrative_scored[narrative_scored["movie_count"] >= 10]

print(f"Narrative keywords scored with 10+ movies: {len(active_narrative):,}")
print()
print("Most POSITIVE (narrative, 10+ movies):")
display(
    active_narrative.nlargest(15, "scl_valence")
    [["name","keyword_type","scl_valence","scl_valence_source","scl_composition_type","movie_count"]]
    .reset_index(drop=True)
)
print("\nMost NEGATIVE (narrative, 10+ movies):")
display(
    active_narrative.nsmallest(15, "scl_valence")
    [["name","keyword_type","scl_valence","scl_valence_source","scl_composition_type","movie_count"]]
    .reset_index(drop=True)
)


Narrative keywords scored with 10+ movies: 9,675

Most POSITIVE (narrative, 10+ movies):


,name,keyword_type,scl_valence,scl_valence_source,scl_composition_type,movie_count
0,empowerment,theme_or_plot,1.000,vad,direct,47
1,pacifism,theme_or_plot,1.000,vad,direct,25
2,tahiti,theme_or_plot,1.000,vad,direct,36
3,mahatma gandhi,theme_or_plot,1.000,unigram_avg,NaN,12
4,centenarian,theme_or_plot,1.000,vad,direct,10
5,entrepreneurship,theme_or_plot,1.000,vad,direct,35
6,female empowerment,theme_or_plot,1.000,unigram_avg,NaN,54
7,humanism,theme_or_plot,1.000,vad,direct,18
8,lighthearted,theme_or_plot,1.000,vad,direct,56
9,heartwarming,theme_or_plot,1.000,vad,direct,24



Most NEGATIVE (narrative, 10+ movies):


,name,keyword_type,scl_valence,scl_valence_source,scl_composition_type,movie_count
0,ku klux klan,plot_event,-1.0,unigram_avg,NaN,40
1,nuclear war,theme_or_plot,-1.0,vad,direct,67
2,cataclysm,theme_or_plot,-1.0,vad,direct,16
3,bulimia,theme_or_plot,-1.0,vad,direct,22
4,pedophilia,theme_or_plot,-1.0,vad,direct,84
5,kleptomania,theme_or_plot,-1.0,vad,direct,10
6,hitman,character_role,-1.0,vad,direct,422
7,sexism,theme_or_plot,-1.0,vad,direct,92
8,sadism,theme_or_plot,-1.0,vad,direct,179
9,necrophilia,theme_or_plot,-1.0,vad,direct,82


## Save

In [12]:
keywords.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(keywords):,} rows → {OUTPUT_FILE}")
print(f"Columns: {list(keywords.columns)}")


Saved 84,842 rows → output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv
Columns: ['tmdb_keyword_id', 'name', 'keyword_type', 'lexical_class', 'min_zipf_freq', 'is_narrative', 'movie_count', 'manually_reviewed', 'scl_valence', 'scl_valence_source', 'scl_in_vad', 'scl_in_opp', 'scl_in_nma', 'scl_composition_type', 'scl_composition_shift', 'scl_shift_zscore', 'scl_shift_percentile', 'scl_is_scored', 'scl_is_phrase_level', 'scl_polarity', 'scl_generated_coverage']


## Upload to Kaggle

In [13]:
import os, shutil

if not os.environ.get("KAGGLE_TOKEN"):
    print("Skipping upload: set KAGGLE_TOKEN env var to upload.")
else:
    import kagglehub
    from datetime import date
    user = kagglehub.whoami(verbose=False)["username"]
    kagglehub.dataset_upload(
        handle=f"{user}/tmdb-keyword-enriched",
        local_dataset_dir=str(KEYWORDS_FILE.parent),
        version_notes=f"Add SCL valence columns — {date.today()}",
        ignore_patterns=["*.json"],
    )
    print(f"→ kaggle.com/datasets/{user}/tmdb-keyword-enriched")


Skipping upload: set KAGGLE_TOKEN env var to upload.
